# 06 - Approach C: Unsupervised Regime Detection (2-state HMM)

**Inertia, Momentum Timing** - Sprint 3, Approach C

*Don't tell the model what a crash is; let it find the regimes.*

- **Framing:** fit a 2-state Gaussian Hidden Markov Model on observable market and momentum state. No supervised crash labels, no predicted-return targets --- the regimes emerge from the data.
- **Observations (3):** `MKT_RF` (market excess return), `mkt_vol6` (market realized vol), `mom_var6` (UMD realized variance).
- **State identification:** the state with higher in-sample average UMD return is the *"normal"* regime; the other is *"stressed."*
- **Filtering:** $P(\text{normal}_t \mid obs_{1..t})$ is computed via the online forward algorithm --- no backward smoothing, no future information.
- **Weight:** $w_t = P(\text{normal}_t \mid obs_{1..t})$. Clean, continuous, no threshold tuning.
- **OOS:** expanding window, HMM refit annually starting 2000-01.
- **Cost:** 20 bps one-way.

**Target to beat:** DM OOS Sharpe 0.37 (notebook 03).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from hmmlearn.hmm import GaussianHMM

from src.features import build_features, DM_FEATURES
from src.backtest import (
    expanding_window_oos, weights_from_predictions, apply_weights,
)
from src.evaluation import perf_table, sharpe_bootstrap_ci, alpha_table, subsample_table
from src.data import get_factor_panel
from src.inertia_style import (
    apply_style, C, FULL_COL,
    save_fig, save_table, legend_below,
)

apply_style()
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

FIG_DIR, TABLE_DIR = '../figures', '../tables'
OOS_START = '2000-01-01'

## 1. Observations panel

In [2]:
df = build_features(include_macro=False)
HMM_OBS = ['MKT_RF', 'mkt_vol6', 'mom_var6']
complete = df[HMM_OBS + ['UMD_next','UMD']].dropna()
print(f'Panel: {complete.shape}, {complete.index.min().date()} -> {complete.index.max().date()}')
print(f'HMM observations: {HMM_OBS}')

Panel: (1178, 5), 1927-12-31 -> 2026-01-31
HMM observations: ['MKT_RF', 'mkt_vol6', 'mom_var6']


## 2. HMM filter class

Wraps a fitted HMM so that `.predict(X)` returns filtered $P(\text{normal} \mid obs_{1..t})$ --- i.e. uses only information up through each OOS month. We cache the posterior at the end of training and advance the forward algorithm one step per OOS observation.

In [3]:
class HMMFilter:
    def __init__(self, hmm, normal_state, train_X):
        self.hmm = hmm
        self.normal_state = normal_state
        post = hmm.predict_proba(train_X)
        self._last_post = post[-1]

    def predict(self, X):
        X = np.asarray(X)
        n = len(X)
        probs = np.zeros(n)
        post = self._last_post.copy()
        transmat = self.hmm.transmat_
        for i in range(n):
            post = post @ transmat                                 # predict (transition)
            log_emit = self.hmm._compute_log_likelihood(X[i:i+1])  # (1, K)
            emit = np.exp(log_emit[0] - log_emit[0].max())         # stable normalization
            post = post * emit
            s = post.sum()
            if s > 0:
                post = post / s
            probs[i] = post[self.normal_state]
        return probs


def fit_hmm(X_train, y_next_umd):
    hmm = GaussianHMM(
        n_components=2, covariance_type='full',
        n_iter=100, random_state=42,
    ).fit(X_train)
    # Identify 'normal' state as the one with higher average UMD_next
    states = hmm.predict(X_train)
    state_returns = []
    for s in range(hmm.n_components):
        mask = (states == s)
        state_returns.append(float(np.mean(y_next_umd[mask])) if mask.any() else -np.inf)
    normal_state = int(np.argmax(state_returns))
    return HMMFilter(hmm, normal_state, X_train)

## 3. Run OOS backtest

In [4]:
p_normal = expanding_window_oos(
    complete, HMM_OBS, 'UMD_next',
    fit_fn=fit_hmm, oos_start=OOS_START,
    refit_months=12, min_train_months=120,
)
print(f'OOS months: {len(p_normal)}')
print(f'P(normal) distribution:')
print(f'  mean={p_normal.mean():.3f}  std={p_normal.std():.3f}  '
      f'min={p_normal.min():.3f}  max={p_normal.max():.3f}')
print(f'  percentiles: 10% {p_normal.quantile(.1):.3f}, 50% {p_normal.quantile(.5):.3f}, 90% {p_normal.quantile(.9):.3f}')

Model is not converging.  Current: 5177.548193547743 is not greater than 5178.6391556520775. Delta is -1.0909621043347215


Model is not converging.  Current: 5208.907806913154 is not greater than 5209.624834014978. Delta is -0.7170271018239873


Model is not converging.  Current: 5238.6891665490375 is not greater than 5239.346810458161. Delta is -0.6576439091231805


Model is not converging.  Current: 5417.977853372703 is not greater than 5418.630888157755. Delta is -0.6530347850521139


Model is not converging.  Current: 5975.188588154833 is not greater than 5975.269745252876. Delta is -0.08115709804314974


...

Model is not converging.  Current: 6608.331676043131 is not greater than 6608.828753987554. Delta is -0.4970779444229265


Model is not converging.  Current: 6723.307470832543 is not greater than 6724.013158601045. Delta is -0.7056877685017753


Model is not converging.  Current: 6814.955569752395 is not greater than 6815.496293300247. Delta is -0.5407235478523944


Model is not converging.  Current: 6906.580509747928 is not greater than 6906.729564000074. Delta is -0.14905425214601564


OOS months: 313
P(normal) distribution:
  mean=0.726  std=0.423  min=0.000  max=1.000
  percentiles: 10% 0.000, 50% 1.000, 90% 1.000


## 4. Weight: $w_t = P(\text{normal}_t)$

Continuous, no threshold, bounded in $[0, 1]$. Directly interpretable: exposure equals the HMM's belief that we are in the good regime.

In [5]:
w_hmm = p_normal.clip(0, 1).rename('weight')
back = apply_weights(w_hmm, df['UMD'], cost_bps_oneway=20.0)
print(f'Weight mean {w_hmm.mean():.2f}, range [{w_hmm.min():.2f}, {w_hmm.max():.2f}]')
print(f'% months with w < 0.1: {(w_hmm < 0.1).mean()*100:.1f}%   % months w > 0.9: {(w_hmm > 0.9).mean()*100:.1f}%')
back.head()

Weight mean 0.73, range [0.00, 1.00]
% months with w < 0.1: 23.3%   % months w > 0.9: 68.1%


,weight,r_next,r_gross,turnover,cost,r_net
date,,,,,,
2000-01-31,0.9994,0.1802,0.1801,0.9994,0.0020,0.1781
2000-02-29,0.8116,-0.0685,-0.0556,0.1878,0.0004,-0.0560
2000-03-31,0.0000,-0.0860,-0.0000,0.8116,0.0016,-0.0016
2000-04-30,0.0000,-0.0899,-0.0000,0.0000,0.0000,-0.0000
2000-05-31,0.0000,0.1663,0.0000,0.0000,0.0000,-0.0000


## 5. Rebuild DM OOS benchmark

In [6]:
class OLSModel:
    def __init__(self, res): self.res = res
    def predict(self, X):
        return self.res.predict(sm.add_constant(X, has_constant='add'))

def fit_ols(X, y):
    return OLSModel(sm.OLS(y, sm.add_constant(X)).fit())

dm_preds = expanding_window_oos(df, DM_FEATURES, 'UMD_next', fit_fn=fit_ols,
                                 oos_start=OOS_START, refit_months=12, min_train_months=120)
dm_w, _ = weights_from_predictions(dm_preds, df['UMD'], cap=(-1.0, 3.0))
dm_back = apply_weights(dm_w, df['UMD'], cost_bps_oneway=20.0)

## 6. Performance comparison

In [7]:
idx = back.index.intersection(dm_back.index)
static_r = df['UMD_next'].reindex(idx)

returns = {
    'Static UMD':         static_r,
    'DM OOS (net)':       dm_back.loc[idx, 'r_net'],
    'Approach C (HMM)':   back.loc[idx, 'r_net'],
}

perf = perf_table(returns)[['n_months','mean_ann','vol_ann','sharpe_ann','skew','excess_kurt','max_drawdown']]
save_table(perf, '06_approach_c_performance', out_dir=TABLE_DIR)
perf

  (skipped tex for 06_approach_c_performance: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).)
  saved: ../tables/06_approach_c_performance.{csv,md}


,n_months,mean_ann,vol_ann,sharpe_ann,skew,excess_kurt,max_drawdown
Static UMD,313,0.0216,0.1750,0.1234,-1.4725,9.4471,-0.5782
DM OOS (net),313,0.0627,0.1695,0.3696,-0.2025,4.9876,-0.2741
Approach C (HMM),313,0.0377,0.1129,0.3338,-0.6178,8.5670,-0.3063


In [8]:
boot = {name: sharpe_bootstrap_ci(r, n_boot=2000) for name, r in returns.items()}
boot_df = pd.DataFrame(boot).T[['sharpe','ci_low','ci_high']]
save_table(boot_df, '06_approach_c_sharpe_bootstrap', out_dir=TABLE_DIR)
boot_df

  (skipped tex for 06_approach_c_sharpe_bootstrap: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).)
  saved: ../tables/06_approach_c_sharpe_bootstrap.{csv,md}


,sharpe,ci_low,ci_high
Static UMD,0.1234,-0.2723,0.5096
DM OOS (net),0.3696,0.0330,0.6387
Approach C (HMM),0.3338,-0.0483,0.6912


## 7. Alphas vs FF5+UMD

In [9]:
factor_panel = get_factor_panel()
alpha_df = alpha_table(
    {k: v for k, v in returns.items() if k != 'Static UMD'},
    factor_panel, spec='FF5_UMD',
)
save_table(alpha_df, '06_approach_c_alpha_ff5umd', out_dir=TABLE_DIR)
alpha_df.T

  (skipped tex for 06_approach_c_alpha_ff5umd: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).)
  saved: ../tables/06_approach_c_alpha_ff5umd.{csv,md}


,DM OOS (net),Approach C (HMM)
alpha_monthly,0.0046,0.0025
alpha_annual,0.0546,0.0304
alpha_t,1.5220,1.3971
alpha_p,0.1280,0.1624
r2,0.0104,0.0125
n_obs,313.0000,313.0000
MKT_RF_beta,-0.0230,-0.0478
MKT_RF_t,-0.2943,-0.9368
SMB_beta,-0.0658,-0.0547
SMB_t,-0.6859,-0.7962


## 8. Subsample Sharpe

In [10]:
splits = {
    '2000-2009':    ('2000-01', '2009-12'),
    '2010-2019':    ('2010-01', '2019-12'),
    '2020-present': ('2020-01', None),
}
sub = subsample_table(returns, splits)
save_table(sub, '06_approach_c_subsample_sharpe', out_dir=TABLE_DIR)
sub

  (skipped tex for 06_approach_c_subsample_sharpe: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).)
  saved: ../tables/06_approach_c_subsample_sharpe.{csv,md}


,2000-2009,2010-2019,2020-present
Static UMD,0.0106,0.3912,0.1302
DM OOS (net),0.4260,0.2734,0.5030
Approach C (HMM),0.2284,0.3462,0.5435


## 9. Cumulative return

In [11]:
plot_df = pd.DataFrame(returns).dropna()
cum = (1 + plot_df).cumprod()

fig, ax = plt.subplots(figsize=(FULL_COL, 3.2))
ax.plot(cum.index, cum['Static UMD'],       color=C['muted'],  linewidth=1.0, label='Static UMD')
ax.plot(cum.index, cum['DM OOS (net)'],     color=C['purple'], linewidth=1.0, label='DM OOS')
ax.plot(cum.index, cum['Approach C (HMM)'], color=C['blue'],   linewidth=1.3, label='Approach C (HMM)')
ax.set_yscale('log')
ax.set_ylabel('Cumulative growth of $1 (log)')
ax.set_title('Approach C (HMM) vs DM OOS vs Static UMD', loc='left', color=C['dark'])
legend_below(ax)
save_fig(fig, '14_approach_c_cumret', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/14_approach_c_cumret.{pdf,png}


## 10. Drawdown

In [12]:
def _dd(r):
    c = (1 + r).cumprod(); return c / c.cummax() - 1

fig, ax = plt.subplots(figsize=(FULL_COL, 2.8))
ax.plot(_dd(returns['Static UMD']).index,   _dd(returns['Static UMD']).values,   color=C['muted'],  linewidth=0.7, label='Static UMD')
ax.plot(_dd(returns['DM OOS (net)']).index, _dd(returns['DM OOS (net)']).values, color=C['purple'], linewidth=0.9, label='DM OOS')
dC = _dd(returns['Approach C (HMM)'])
ax.fill_between(dC.index, dC.values, 0, color=C['blue'], alpha=0.18, linewidth=0)
ax.plot(dC.index, dC.values, color=C['blue'], linewidth=0.9, label='Approach C (HMM)')
ax.axhline(0, color=C['dark'], linewidth=0.3)
ax.set_ylabel('Drawdown')
ax.set_title('Drawdowns: Approach C (HMM) vs DM OOS vs Static UMD', loc='left', color=C['dark'])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
legend_below(ax)
save_fig(fig, '15_approach_c_drawdown', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/15_approach_c_drawdown.{pdf,png}


## 11. P(normal) timeline

Reveals when the HMM thinks we are in the stressed regime. Strong bimodal behavior (values near 0 or 1, with few in the middle) is characteristic of a well-specified HMM.

In [13]:
fig, ax = plt.subplots(figsize=(FULL_COL, 2.8))
ax.plot(p_normal.index, p_normal.values, color=C['blue'], linewidth=0.7, label='P(normal | obs up to t)')
stressed = (p_normal < 0.5)
ax.fill_between(p_normal.index, 0, stressed.astype(float),
                where=stressed, color=C['red'], alpha=0.15, linewidth=0,
                label='Stressed regime (p < 0.5)')
ax.set_ylabel('P(normal)')
ax.set_ylim(-0.02, 1.02)
ax.set_title('Approach C: filtered probability of the normal regime', loc='left', color=C['dark'])
legend_below(ax)
save_fig(fig, '16_approach_c_pnormal_timeseries', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/16_approach_c_pnormal_timeseries.{pdf,png}


## 12. HMM parameter inspection (diagnostic)

Full-sample fit so we can inspect what the two states look like. Not part of the OOS backtest.

In [14]:
hmm_full = GaussianHMM(n_components=2, covariance_type='full',
                       n_iter=100, random_state=42).fit(complete[HMM_OBS].values)
states = hmm_full.predict(complete[HMM_OBS].values)
state_returns = [float(np.mean(complete['UMD_next'].values[states == s])) * 12
                 for s in range(hmm_full.n_components)]
normal = int(np.argmax(state_returns))

# Mean vectors and annualized UMD returns per state
params = pd.DataFrame({
    'state':            [f'State {s}' for s in range(hmm_full.n_components)],
    'label':            ['normal' if s == normal else 'stressed' for s in range(hmm_full.n_components)],
    'mkt_ret_mean':     [hmm_full.means_[s, 0]   for s in range(hmm_full.n_components)],
    'mkt_vol6_mean':    [hmm_full.means_[s, 1]   for s in range(hmm_full.n_components)],
    'mom_var6_mean':    [hmm_full.means_[s, 2]   for s in range(hmm_full.n_components)],
    'umd_ann_return':   state_returns,
    'stay_prob':        [hmm_full.transmat_[s, s] for s in range(hmm_full.n_components)],
})
save_table(params.set_index('state'), '06_approach_c_hmm_parameters', out_dir=TABLE_DIR)
params

Model is not converging.  Current: 6912.398301758058 is not greater than 6912.600964507287. Delta is -0.20266274922960292


  (skipped tex for 06_approach_c_hmm_parameters: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).)
  saved: ../tables/06_approach_c_hmm_parameters.{csv,md}


,state,label,mkt_ret_mean,mkt_vol6_mean,mom_var6_mean,umd_ann_return,stay_prob
0,State 0,stressed,0.0030,0.2719,0.0923,-0.0395,0.9173
1,State 1,normal,0.0077,0.1223,0.0092,0.1026,0.9779


## Takeaways --- Approach C

- A 2-state HMM fit only on three observable series (market return, market vol, UMD vol) recovers a clean regime structure: one state has much higher UMD returns and lower market vol; the other is a high-volatility stressed state that UMD underperforms in.
- The filtered $P(\text{normal}_t)$ is strongly bimodal --- the model is usually very confident which regime we're in.
- Using $P(\text{normal})$ directly as the weight gives a Sharpe comparable to DM OOS **without any supervised labels or hand-crafted features** --- just the three raw observations.
- The HMM transition probabilities (each state has high self-persistence) match the intuition that market regimes are sticky.

**For the prospectus:**
- **Strength:** no arbitrary crash thresholds; regimes are data-driven.
- **Weakness:** 2-state HMM is a strong assumption; markets arguably have more than two regimes (3-state variants exist).
- **Honest:** we chose the simpler model to avoid overfitting on the relatively short OOS window, and because the class's ML chapter warns against complex models in low signal-to-noise settings.